# Phase 10 — Launch the full-system Streamlit UI on Colab

Wires Phases 5 (disease) + 6 (soil) + 7 (RAG) + 8 (integration) + 9 (explainability) into a single live demo.

Run the cells in order. The last cell starts streamlit + a localtunnel and prints a public URL you can open in any browser. URL is session-bound — Colab session timeout kills it.

**Requirements:** T4 GPU runtime, HF token (for the private corpus + Llama-3.1-8B).

## 1. Install runtime deps + clone the repo

In [ ]:
import os, subprocess, sys

DEPS = [
    "streamlit==1.39.0",
    "transformers==4.45.2",
    "accelerate==0.34.2",
    "bitsandbytes==0.44.1",
    "sentence-transformers==3.0.1",
    "chromadb==0.5.5",
    "datasets==2.21.0",
    "huggingface_hub==0.25.1",
    "timm==1.0.9",
    "rembg==2.0.59",
    "grad-cam==1.5.4",
    "pillow",
    "numpy<2",
    "opencv-python-headless",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS], check=True)
print("deps installed")

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/ANKIT-PAWAR/Thesis.git"  # adjust if your remote differs
REPO_DIR = "/content/Thesis"
BRANCH = "cleanup/pdf-alignment"  # the working branch

if not os.path.isdir(REPO_DIR):
    # GIT_LFS_SKIP_SMUDGE=1 dodges the LFS bandwidth quota on big checkpoints
    env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR],
        check=True, env=env,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 2. HuggingFace auth (private corpus + Llama-3.1-8B)

In [ ]:
from huggingface_hub import login
from google.colab import userdata  # type: ignore

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
print("HF login complete")

## 3. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "No CUDA — switch the runtime to T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f"VRAM free: {free/2**30:.2f} GB / {total/2**30:.2f} GB")

## 4. Start Streamlit + localtunnel; print the public URL

Streamlit runs in a background process on port 8501. `localtunnel` exposes it to a public `https://*.loca.lt` URL. The URL prints below — open it in any browser. localtunnel shows a one-time "Continue" gate that wants the **tunnel password** (your Colab instance's public IP); we print that too.

Note: the session dies on Colab timeout (expected for a demo).

In [ ]:
import subprocess, time, urllib.request

# (a) install localtunnel via npm — Colab has node preinstalled
subprocess.run(["npm", "install", "-g", "localtunnel"], check=True)

# (b) start streamlit (background; logs to /content/streamlit.log)
streamlit_proc = subprocess.Popen(
    [
        "streamlit", "run", "app/streamlit_app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--browser.gatherUsageStats=false",
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)
print("streamlit pid:", streamlit_proc.pid)

# (c) wait until streamlit is reachable on 8501
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8501", timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("streamlit didn't come up — see /content/streamlit.log")
print("streamlit is up on port 8501")

# (d) start localtunnel; capture URL
lt_proc = subprocess.Popen(
    ["lt", "--port", "8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
for _ in range(30):
    line = lt_proc.stdout.readline()
    if not line:
        time.sleep(1); continue
    print(line.rstrip())
    if "loca.lt" in line:
        public_url = line.strip().split()[-1]
        break
assert public_url, "localtunnel didn't print a URL — re-run this cell."

# (e) print the tunnel password (= public IP)
ip = urllib.request.urlopen("https://loca.lt/mytunnelpassword", timeout=10).read().decode().strip()
print("\n===== OPEN THIS URL =====")
print(public_url)
print("Tunnel password (enter on the loca.lt gate):", ip)
print("==========================\n")

## 5. (Optional) tail the streamlit log

If the app errors in the browser, run this to see the server-side traceback.

In [ ]:
!tail -n 80 /content/streamlit.log